In [ ]:
import pdfplumber
import os


In [ ]:
os.chdir("E:\Projects\Agentic FinTracker")

In [ ]:
os.getcwd()

In [ ]:
pdf_path='./data/bank_statement (1).pdf'
os.chdir("E:\Projects\Agentic FinTracker")

In [ ]:
with pdfplumber.open(pdf_path) as pdf:
    text = []
    tables=[]
    for page in pdf.pages:
        if page.extract_tables():
            for table in page.extract_tables():
                tables.append(table)
        else:
            text.append(page.extract_text())
        
    print(tables)

In [ ]:
def pdf_to_text(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        response = None
        context = []
        for page in pdf.pages:
            if page.extract_tables():
                for table in page.extract_tables():
                    context.append(table)
            else:
                context.append(page.extract_text())
        if context:
            
            response = context
            
    return response

In [ ]:
from config.config import Config
api_key = Config.API_KEY

In [ ]:
context = pdf_to_text(pdf_path)

In [ ]:
print(context)

In [ ]:
from openai import OpenAI
def query_chatgpt(context):
    client = OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": "You are a text extractor. most of the time you will be given a bank statement in text format and you must extract the relevant information remove unnecessary information most of the given data is in tabular format.if it is a text base output convert into json format.when you give the otput you must check if  the output in json format if not you must convert it to json format and give the output in json format and dont give any comments or questions just give the json output."},
            {"role": "user", "content": f"Context: {context}"}
        ]
    )
    raw = response.choices[0].message.content
    return raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()

In [ ]:
from IPython.display import JSON, display
response=query_chatgpt(pdf_to_text(pdf_path))
print(response)

In [ ]:
import json
json_response = json.dumps((response), indent=4)
print(json_response)

In [ ]:
display(JSON(response))

In [ ]:
from app.backend.models.chatbot import Chatbot
from config.config import Config
from app.backend.database.db_manager import DatabaseManager
from app.backend.tools.db_tools import DBTools

pdf_path='./data/bank_statement (1).pdf'

In [ ]:
import os
os.chdir("E:\Projects\Agentic FinTracker")

In [ ]:
os.getcwd()

In [ ]:
pdf_path

In [ ]:
db = DatabaseManager(Config.DATABASE_PATH)
db_tools=DBTools(database_obj=db)
tools=db_tools.functions_formatter()
chatbot=Chatbot(api_key=Config.API_KEY,tools_obj=db_tools)

In [ ]:
response = chatbot.text_to_table(chatbot.pdf_to_text(pdf_path))

In [ ]:
print(response)

In [1]:
import os
pdf_path='./data/bank_statement (1).pdf'
os.chdir("E:\Projects\Agentic FinTracker")

<>:3: SyntaxWarning: invalid escape sequence '\P'
<>:3: SyntaxWarning: invalid escape sequence '\P'
C:\Users\sharu\AppData\Local\Temp\ipykernel_9888\4284637688.py:3: SyntaxWarning: invalid escape sequence '\P'
  os.chdir("E:\Projects\Agentic FinTracker")


In [2]:
os.getcwd()

'E:\\Projects\\Agentic FinTracker'

In [3]:
from app.backend.models.chatbot import Chatbot
from config.config import Config
from app.backend.database.db_manager import DatabaseManager
from app.backend.tools.db_tools import DBTools

e:\Projects\Agentic FinTracker\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
system_prompt="""
You are helpfull personal financial assistant.
"""

In [3]:
system_prompt

"You are a personal financial assistant. Help users track their income and expenses using the available tools. Today's date is 2026-04-30.\n\nYou can:\n- Log salary: use set_salary tool\n- Log expenses: use log_expense tool  \n- Check balance: use get_balance tool\n- Get expense summary: use get_expense_summary tool\n\nWhen a user uploads a bank statement PDF, the extracted data will be provided to you in JSON format with salary and expenses. Automatically log each record one by one using the appropriate tools without asking for confirmation. After logging all records, give the user a summary of what was logged.\n\nWhen logging expenses use only these categories: Food, Transport, Utilities, Rent, Healthcare, Shopping, Entertainment, Education, Insurance, Subscriptions, Fuel, Other.\n\nAlways be concise and friendly. If the user asks anything unrelated to personal finance, politely let them know you can only help with financial tracking."

In [4]:
db = DatabaseManager(Config.DATABASE_PATH)
db_tools = DBTools(database_obj=db)
tools = db_tools.functions_formatter()
chatbot = Chatbot(api_key=Config.API_KEY, tools_obj=db_tools)

calculate balance function
